# Testing DFT in CKKS to study OpenFHE and CKKS implementation

In [4]:
import openfhe as fhe
import numpy as np

In [5]:
class CKKSRunner:
    def __init__(self, batchSize: int = 8, scaleModSize: int = 50, depth: int = 1):
        scaleModSize = scaleModSize     # delta
        self.batchSize = batchSize           # N/2
        self.depth = depth                   # multiplicative depth

        self.params = fhe.CCParamsCKKSRNS()

        self.params.SetBatchSize(self.batchSize)
        self.params.SetScalingModSize(scaleModSize)
        self.params.SetMultiplicativeDepth(self.depth)
        self.params.SetCKKSDataType(fhe.CKKSDataType.COMPLEX)

        self.cc = fhe.GenCryptoContext(self.params)

    def generateDFTmatrix(self, N: int):

        n = np.arange(N)
        k = n.reshape((N, 1))

        W = np.exp(-2j * np.pi * k * n / N)

        return W

    # Halevi and Shoup method for matrix multiplication with 1 level consumption
    # We need a way to diagonalize the matrix
    # Algorith 2 from HyDia

    def diagonalizeMatrix(self, matrix: np.ndarray) -> np.ndarray:
        K = len(matrix)
        N = len(matrix[0])

        diagonals = np.zeros((N, K), dtype=complex)

        for i in range(N):
            for j in range(K):
                colIndex = (j + i) % N
                diagonals[i][j] = matrix[j][colIndex]

        return diagonals

    def matrixVectMult(self, W: np.ndarray, x_ct: fhe.Ciphertext):

        diagW = self.diagonalizeMatrix(W)

        result_ct = None
        for k in range(len(W)):
            diagW_k = diagW[k]
            diagW_k_pt = self.cc.MakeCKKSPackedPlaintext(diagW_k)

            if (k == 0):
                rotated_ct = x_ct
            else:
                rotated_ct = self.cc.EvalRotate(x_ct, k)

            term_ct = self.cc.EvalMult(rotated_ct, diagW_k_pt)

            if result_ct is None:
                result_ct = term_ct
            else:
                result_ct = self.cc.EvalAdd(result_ct, term_ct)

        return result_ct



    def naiveHDFT(self, x):
        N = len(x)

        self.cc.Enable(fhe.PKESchemeFeature.PKE)
        self.cc.Enable(fhe.PKESchemeFeature.KEYSWITCH)
        self.cc.Enable(fhe.PKESchemeFeature.LEVELEDSHE)

        keys = self.cc.KeyGen()

        # precisamos gerar as chaves de rotação para k = 1 ... N-1
        # pois o CKKS faz rotação com KeySwitching
        rotations = list(range(1, N))
        self.cc.EvalRotateKeyGen(keys.secretKey, rotations)

        x_pt = self.cc.MakeCKKSPackedPlaintext(x)

        x_ct = self.cc.Encrypt(keys.publicKey, x_pt)

        W = self.generateDFTmatrix(N)

        result_ct = self.matrixVectMult(W, x_ct)

        result_pt: fhe.Plaintext = self.cc.Decrypt(keys.secretKey, result_ct)
        result_pt.SetLength(N)
        result = np.array(result_pt.GetCKKSPackedValue())

        expected = W @ x

        # print(f"Resultado HE:       {result}")
        # print(f"Esperado:           {expected}")
        print(f"Erro máximo entre resultado homomorfico e esperado:        {np.max(np.abs(result - expected)):.2e}")




In [6]:
def main():
    test_cases = [
        # (N, descrição)
        [1.0 + 3j, 5.0 + 0j, 3.0 + 0j, 4.0 + 0j],                          # N=4   - baseline
        [1.0+1j, 2.0+0j, 3.0+1j, 4.0+0j, 5.0+2j, 6.0+0j, 7.0+1j, 8.0+0j], # N=8   - 2x baseline
        [complex(i, i % 3) for i in range(16)],                               # N=16  - começa a pesar
        [complex(i, i % 3) for i in range(32)],                               # N=32  - pesado
        [complex(i, i % 3) for i in range(64)],                               # N=64  - limite seguro p/ 8GB
    ]

    for i, input_vec in enumerate(test_cases):
        N = len(input_vec)
        print(f"\n{'='*70}")
        print(f"Test {i+1}: N={N}")

        ckksRunner = CKKSRunner(
            batchSize=N,
            scaleModSize=50,
            depth=1
        )

        # %timeit -n 1 -r 3 ckksRunner.naiveHDFT(input_vec)
        ckksRunner.naiveHDFT(input_vec)

main()


Test 1: N=4
Erro máximo entre resultado homomorfico e esperado:        1.24e-13

Test 2: N=8
Erro máximo entre resultado homomorfico e esperado:        4.89e-13

Test 3: N=16
Erro máximo entre resultado homomorfico e esperado:        1.32e-12

Test 4: N=32
Erro máximo entre resultado homomorfico e esperado:        2.58e-12

Test 5: N=64
Erro máximo entre resultado homomorfico e esperado:        5.78e-12
